# Gene-Expression Phenotype Classification

Leakage-controlled ML pipeline (Logistic Regression, LASSO, Elastic Net, KNN, Random Forest, PCA + LR) with MCC threshold optimisation and subgroup analysis.

Works on **any** binary phenotype from a gene-expression dataset: the bundled GSE2034 files, any NCBI GEO series (downloaded automatically), or your own CSV files.

**On Google Colab:** run the first cell to get the code. Locally, start from the second cell.

In [ ]:
# Colab only: download the project and move into it
!git clone https://github.com/moudi-eng/gene-expression-classification.git
%cd gene-expression-classification

In [ ]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')  # run everything from the repository root
!pip install -q -r requirements.txt

## 1. Reproduce the report (GSE2034, breast-cancer relapse)

Expected: best model = LASSO, test MCC ≈ 0.268, ROC-AUC ≈ 0.610.

In [ ]:
!python src/pipeline.py --config configs/gse2034.json

In [ ]:
import pandas as pd
from IPython.display import Image, display

def show(name, figures=('roc_curves', 'metric_comparison', 'subgroup_performance', 'top_features')):
    out = f'results/{name}'
    display(pd.read_csv(f'{out}/model_comparison.csv')[
        ['Model', 'ROC_AUC', 'PR_AUC', 'MCC', 'Sensitivity', 'Specificity', 'F1', 'Threshold']].round(3))
    if os.path.exists(f'{out}/subgroup_analysis.csv'):
        display(pd.read_csv(f'{out}/subgroup_analysis.csv')[
            ['Subgroup', 'N', 'ROC_AUC', 'MCC', 'Sensitivity', 'Specificity']].round(3))
    for fig in figures:
        path = f'{out}/figures/{fig}.png'
        if os.path.exists(path):
            display(Image(path, width=600))

show('GSE2034')

## 2. Another GEO dataset (GSE7390, distant metastasis)

Downloaded from NCBI GEO on first run (~45 MB). Gene symbols are added to the top features automatically.

In [ ]:
!python src/pipeline.py --config configs/gse7390_example.json
show('GSE7390')

## 3. Your own dataset

1. List the clinical columns of a GEO series (or of your own files).
2. Pick the target column and the value that counts as the positive class.

In [ ]:
GEO_ID = 'GSE7390'   # change me
!python src/pipeline.py --geo {GEO_ID} --list-columns

In [ ]:
TARGET, POSITIVE, SUBGROUP = 'er', '1', 'grade'   # change me
!python src/pipeline.py --geo {GEO_ID} --target {TARGET} --positive {POSITIVE} --subgroup {SUBGROUP} --name {GEO_ID}_{TARGET}
show(f'{GEO_ID}_{TARGET}')